# Instagram Usage & Psychological Risk — Feature Engineering

This notebook:
1. Loads and combines both datasets
2. Creates a composite psychological-risk target
3. Identifies and **removes highly correlated features** (keeping the one more predictive of the target)
4. Ranks remaining features via multiple selection methods (MI, Spearman, ExtraTrees, LightGBM)
5. Produces an ensemble Borda-count ranking

All plots → `../results/feature_engineering/`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import ExtraTreesRegressor
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.edgecolor': '#CCCCCC',
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.9,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': '#CCCCCC',
    'axes.spines.bottom': '#CCCCCC',
})

PALETTE  = sns.color_palette('Set2')
PALETTE2 = sns.color_palette('tab10')

TARGET_RAW  = 'perceived_stress_score'
TARGET_COMP = 'psychological_risk'   # composite z(stress) - z(happiness)
CORR_THRESH = 0.85                   # drop one feature if |r| > this

RESULTS = Path('../results/feature_engineering')
RESULTS.mkdir(parents=True, exist_ok=True)
print(f'Saving plots to: {RESULTS.resolve()}')

## 1. Load & Combine Both Datasets

In [ ]:
print('Loading dataset 1 ...')
df1 = pd.read_csv('../dataset/instagram_usage_lifestyle.csv', low_memory=False)
print(f'  instagram_usage_lifestyle  : {df1.shape}')

print('Loading dataset 2 ...')
df2 = pd.read_csv('../dataset/instagram_users_lifestyle.csv', low_memory=False)
print(f'  instagram_users_lifestyle  : {df2.shape}')

df_full = pd.concat([df1, df2], ignore_index=True)
if 'user_id' in df_full.columns:
    df_full = df_full.drop_duplicates(subset='user_id')
else:
    df_full = df_full.drop_duplicates()

print(f'Combined (deduplicated): {df_full.shape}')

# Sample 150 K for feature engineering (balance speed vs coverage)
SAMPLE = min(150_000, len(df_full))
df = df_full.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
print(f'Working sample          : {df.shape}')

## 2. Create Composite Target & Encode Categoricals

In [ ]:
# Composite target: high score = high stress AND low happiness
if 'self_reported_happiness' in df.columns:
    z_stress = stats.zscore(df[TARGET_RAW].fillna(df[TARGET_RAW].median()))
    z_happy  = stats.zscore(df['self_reported_happiness'].fillna(
                              df['self_reported_happiness'].median()))
    df[TARGET_COMP] = z_stress - z_happy
    print('Composite target created: psychological_risk = z(stress) - z(happiness)')
else:
    df[TARGET_COMP] = stats.zscore(df[TARGET_RAW].fillna(df[TARGET_RAW].median()))
    print('Composite target = z(stress)  [happiness not found]')

# Drop columns that carry no predictive signal
drop_cols = ['user_id', 'app_name', 'country', 'last_login_date',
             'content_type_preference', 'preferred_content_theme',
             TARGET_RAW, 'self_reported_happiness']

df_enc = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore').copy()

# Ordinal encode string features
le = LabelEncoder()
for col in df_enc.select_dtypes(include=['object', 'category']).columns:
    if col != TARGET_COMP:
        df_enc[col] = le.fit_transform(df_enc[col].fillna('Unknown').astype(str))

# Boolean → int
for col in df_enc.select_dtypes(include='bool').columns:
    df_enc[col] = df_enc[col].astype(int)

df_enc = df_enc.fillna(df_enc.median(numeric_only=True))

target_cols = [TARGET_COMP]
feature_cols = [c for c in df_enc.columns if c not in target_cols]

print(f'\nFeature matrix: {len(feature_cols)} features, {len(df_enc):,} rows')
print(f'Features: {feature_cols}')

## 3. Full Correlation Matrix

In [ ]:
corr_full = df_enc[feature_cols].corr().abs()
mask_triu = np.triu(np.ones_like(corr_full, dtype=bool))

cmap = sns.diverging_palette(220, 20, as_cmap=True)

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(corr_full, mask=mask_triu, cmap='Reds', vmin=0, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'shrink': 0.75},
            annot=False, ax=ax)
ax.set_title(f'Absolute Pearson Correlation Matrix — All {len(feature_cols)} Features\n'
             f'(Red = high correlation, potential redundancy)', pad=15)
fig.tight_layout()
fig.savefig(RESULTS / 'correlation_matrix_before.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correlation_matrix_before.png')

## 4. Identify & Remove Highly Correlated Feature Pairs

In [ ]:
# Find all pairs with |r| > CORR_THRESH
high_corr_pairs = []
cols = feature_cols
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr_full.iloc[i, j]
        if r > CORR_THRESH:
            high_corr_pairs.append({
                'feature_A': cols[i],
                'feature_B': cols[j],
                'abs_correlation': round(r, 4)
            })

hc_df = pd.DataFrame(high_corr_pairs).sort_values('abs_correlation', ascending=False)
print(f'Found {len(hc_df)} highly correlated pairs  (threshold = {CORR_THRESH})')
if len(hc_df):
    display(hc_df.reset_index(drop=True))

In [ ]:
# For each pair, keep the feature with HIGHER |correlation to target|; drop the other
y = df_enc[TARGET_COMP].values

features_to_drop = set()
drop_log = []

for _, row in hc_df.iterrows():
    fA, fB = row['feature_A'], row['feature_B']
    if fA in features_to_drop or fB in features_to_drop:
        continue  # one of the pair already scheduled for removal
    rA = abs(np.corrcoef(df_enc[fA].values, y)[0, 1])
    rB = abs(np.corrcoef(df_enc[fB].values, y)[0, 1])
    drop  = fB if rA >= rB else fA
    keep  = fA if rA >= rB else fB
    features_to_drop.add(drop)
    drop_log.append({
        'kept': keep, 'dropped': drop,
        'r_kept': round(max(rA, rB), 4),
        'r_dropped': round(min(rA, rB), 4),
        'pair_corr': row['abs_correlation']
    })

drop_log_df = pd.DataFrame(drop_log)
print(f'Dropping {len(features_to_drop)} redundant features:')
if len(drop_log_df):
    display(drop_log_df.reset_index(drop=True))

remaining_features = [f for f in feature_cols if f not in features_to_drop]
print(f'\nFeatures remaining: {len(remaining_features)}')

In [ ]:
# Visualise which features were dropped and why
if len(drop_log_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(drop_log_df) * 0.45)))
    fig.suptitle(f'Features Removed Due to High Correlation (|r| > {CORR_THRESH})')

    # Pair correlation
    y_pos = np.arange(len(drop_log_df))
    axes[0].barh(y_pos, drop_log_df['pair_corr'], color=PALETTE[1], edgecolor='white', height=0.6)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(drop_log_df['dropped'], fontsize=8)
    axes[0].axvline(CORR_THRESH, color='#C0392B', lw=1.5, linestyle='--',
                    label=f'Threshold = {CORR_THRESH}')
    axes[0].set_xlabel('|Correlation| with Paired Feature')
    axes[0].set_title('Removed Feature vs Kept Partner\n(correlation between the pair)')
    axes[0].legend()
    axes[0].invert_yaxis()

    # Target correlation comparison
    width = 0.35
    axes[1].barh(y_pos - width/2, drop_log_df['r_kept'],   height=width,
                 color=PALETTE[2], edgecolor='white', label='Kept feature')
    axes[1].barh(y_pos + width/2, drop_log_df['r_dropped'], height=width,
                 color=PALETTE[3], edgecolor='white', label='Dropped feature')
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(drop_log_df['kept'], fontsize=8)
    axes[1].set_xlabel('|Correlation| with Target (psychological_risk)')
    axes[1].set_title('Target Correlation: Kept vs Dropped')
    axes[1].legend()
    axes[1].invert_yaxis()

    fig.tight_layout()
    fig.savefig(RESULTS / 'dropped_correlated_features.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: dropped_correlated_features.png')
else:
    print('No features were removed (no pairs above threshold).')

## 5. Correlation Matrix After Removing Redundant Features

In [ ]:
corr_after = df_enc[remaining_features].corr().abs()
mask_after  = np.triu(np.ones_like(corr_after, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr_after, mask=mask_after, cmap='Reds', vmin=0, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'shrink': 0.75},
            annot=False, ax=ax)
ax.set_title(f'Correlation Matrix After Removing Redundant Features\n'
             f'({len(remaining_features)} features remain)', pad=15)
fig.tight_layout()
fig.savefig(RESULTS / 'correlation_matrix_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correlation_matrix_after.png')

## 6. Feature Selection — Mutual Information

In [ ]:
X = df_enc[remaining_features].values
y = df_enc[TARGET_COMP].values

print('Computing Mutual Information scores ...')
mi_raw = mutual_info_regression(X, y, random_state=42, n_neighbors=5)
mi_series = pd.Series(mi_raw, index=remaining_features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(mi_series) * 0.28)))
y_pos = np.arange(len(mi_series))
ax.barh(y_pos, mi_series.values, color=PALETTE2[0], edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(mi_series.index, fontsize=8)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Importance — Mutual Information\n(captures nonlinear relationships with target)')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULTS / 'feature_importance_mutual_info.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 10 by MI:')
print(mi_series.head(10).round(4).to_string())

## 7. Feature Selection — Spearman Correlation with Target

In [ ]:
print('Computing Spearman correlations ...')
spear_scores = {}
for feat in remaining_features:
    rho, _ = stats.spearmanr(df_enc[feat].values, y)
    spear_scores[feat] = abs(rho)
spear_series = pd.Series(spear_scores).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(spear_series) * 0.28)))
colors = [PALETTE[0] if v >= 0 else PALETTE[3] for v in spear_series.values]
y_pos = np.arange(len(spear_series))
ax.barh(y_pos, spear_series.values, color=PALETTE2[1], edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(spear_series.index, fontsize=8)
ax.set_xlabel('|Spearman ρ| with Psychological Risk')
ax.set_title('Feature Importance — Spearman Rank Correlation\n(monotonic relationship with target)')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULTS / 'feature_importance_spearman.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 10 by |Spearman ρ|:')
print(spear_series.head(10).round(4).to_string())

## 8. Feature Selection — Extra Trees

In [ ]:
print('Fitting ExtraTreesRegressor ...')
et = ExtraTreesRegressor(n_estimators=150, random_state=42, n_jobs=-1)
et.fit(X, y)
et_series = pd.Series(et.feature_importances_, index=remaining_features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(et_series) * 0.28)))
y_pos = np.arange(len(et_series))
ax.barh(y_pos, et_series.values, color=PALETTE2[2], edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(et_series.index, fontsize=8)
ax.set_xlabel('Feature Importance (impurity-based)')
ax.set_title('Feature Importance — Extra Trees Regressor\n(ensemble tree-based importance)')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULTS / 'feature_importance_extratrees.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 10 by ExtraTrees:')
print(et_series.head(10).round(4).to_string())

## 9. Feature Selection — LightGBM (Gain-based)

In [ ]:
try:
    import lightgbm as lgb
    print('Fitting LightGBM ...')
    lgb_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                   random_state=42, n_jobs=-1, verbose=-1)
    lgb_model.fit(X, y)
    lgb_series = pd.Series(lgb_model.feature_importances_,
                           index=remaining_features).sort_values(ascending=False)
    lgb_available = True
    print('LightGBM fitted successfully.')
except ImportError:
    print('LightGBM not installed — skipping.')
    lgb_series = None
    lgb_available = False

if lgb_available:
    fig, ax = plt.subplots(figsize=(10, max(6, len(lgb_series) * 0.28)))
    y_pos = np.arange(len(lgb_series))
    ax.barh(y_pos, lgb_series.values, color=PALETTE2[3], edgecolor='white', height=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(lgb_series.index, fontsize=8)
    ax.set_xlabel('Feature Importance (gain)')
    ax.set_title('Feature Importance — LightGBM Gain\n(gradient boosting, gain-based split importance)')
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(RESULTS / 'feature_importance_lightgbm.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Top 10 by LightGBM gain:')
    print(lgb_series.head(10).round(2).to_string())

## 10. Borda-Count Ensemble Ranking

In [ ]:
# Collect all method series
methods = {
    'Mutual Information': mi_series,
    'Spearman |ρ|':       spear_series,
    'Extra Trees':        et_series,
}
if lgb_available:
    methods['LightGBM Gain'] = lgb_series

# Convert each method's scores to ranks (rank 1 = best)
rank_df = pd.DataFrame(index=remaining_features)
for method_name, series in methods.items():
    rank_df[method_name] = series.rank(ascending=False)

# Borda score = mean rank across methods (lower = better)
rank_df['Borda Score'] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values('Borda Score')

print('Borda Ensemble Ranking (top 20):')
display(rank_df.head(20).round(2))

rank_df.to_csv(RESULTS / 'feature_rankings_all_methods.csv')
print('\nFull rankings saved: feature_rankings_all_methods.csv')

## 11. Rank Heatmap — Method Agreement

In [ ]:
TOP_N = 20
top_features_borda = rank_df.head(TOP_N).index.tolist()

heat_data = rank_df.loc[top_features_borda, list(methods.keys())]

fig, ax = plt.subplots(figsize=(max(8, len(methods) * 2.2), max(8, TOP_N * 0.55)))
sns.heatmap(heat_data, cmap='RdYlGn_r', annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': 'Rank (1 = most important)', 'shrink': 0.8},
            ax=ax)
ax.set_title(f'Per-Method Feature Ranks — Top {TOP_N} Features (by Borda)', pad=15)
ax.set_ylabel('Feature')
ax.set_xlabel('Selection Method')
plt.xticks(rotation=25, ha='right')
fig.tight_layout()
fig.savefig(RESULTS / 'rank_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: rank_heatmap.png')

## 12. Final Feature Importance — Borda Bar Chart

In [ ]:
# Invert Borda score so longer bar = more important
borda_plot = rank_df.head(TOP_N).copy()
max_rank = len(remaining_features)
borda_plot['Importance Score'] = max_rank - borda_plot['Borda Score']

fig, ax = plt.subplots(figsize=(11, max(7, TOP_N * 0.45)))
y_pos = np.arange(TOP_N)

bars = ax.barh(y_pos, borda_plot['Importance Score'].values,
               color=PALETTE2[:TOP_N] * (TOP_N // 10 + 1),
               edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(borda_plot.index.tolist(), fontsize=9)
ax.set_xlabel('Borda Importance Score  (higher = more important)')
ax.set_title(f'Top {TOP_N} Features — Borda-Count Ensemble Ranking\n'
             f'(aggregated from {len(methods)} feature selection methods)')
ax.invert_yaxis()

# Add rank labels
for i, (bar, borda) in enumerate(zip(bars, borda_plot['Borda Score'].values)):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'#{i+1}  (mean rank {borda:.1f})',
            va='center', fontsize=7.5, color='#333333')

ax.set_xlim(0, borda_plot['Importance Score'].max() * 1.25)

fig.tight_layout()
fig.savefig(RESULTS / 'top_features_borda_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: top_features_borda_ranking.png')

# Export top feature list
top_feature_list = borda_plot.index.tolist()
with open(RESULTS / 'top_features.txt', 'w') as f:
    f.write('\n'.join(top_feature_list))
print(f'Top {TOP_N} features saved to top_features.txt')

## 13. Pairplot of Top 6 Features

In [ ]:
top6 = top_features_borda[:6]
sub6 = df_enc[top6 + [TARGET_COMP]].sample(n=min(5000, len(df_enc)), random_state=1)

# Bin target for colour coding
sub6['Risk Level'] = pd.qcut(sub6[TARGET_COMP], q=3,
                              labels=['Low', 'Medium', 'High'])

g = sns.pairplot(sub6, vars=top6, hue='Risk Level',
                 palette={'Low': PALETTE[2], 'Medium': PALETTE[1], 'High': PALETTE[3]},
                 plot_kws={'alpha': 0.3, 's': 10},
                 diag_kind='kde')
g.figure.suptitle('Pairwise Relationships — Top 6 Features\n(coloured by Psychological Risk Level)',
                   y=1.01)

g.figure.savefig(RESULTS / 'top6_pairplot.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: top6_pairplot.png')

## 14. Summary

In [ ]:
print('=' * 60)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'Original features      : {len(feature_cols)}')
print(f'Removed (high corr.)   : {len(features_to_drop)}  (|r| > {CORR_THRESH})')
print(f'Features after removal : {len(remaining_features)}')
print(f'Selection methods used : {list(methods.keys())}')
print(f'\nTop {TOP_N} features (Borda rank):')
for rank, feat in enumerate(top_feature_list, 1):
    borda_val = rank_df.loc[feat, 'Borda Score']
    print(f'  {rank:2d}. {feat}  (mean rank {borda_val:.2f})')
print('\nAll plots saved to:', RESULTS.resolve())